In [6]:
import pandas as pd
import requests
import time

stations = {
    "ALGORTA_BBIZI2": (43.362056, -3.022782),
    "BARAKALDO": (43.296, -2.989),
    "BASAURI": (43.239, -2.885),
    "ERANDIO": (43.307, -2.973),
    "MAZARREDO": (43.263, -2.935),
    "MUSKIZ": (43.323, -3.113),
    "SANTURTZI": (43.323, -3.032)
}

for name, (lat, lon) in stations.items():
    print(f"Processing: {name}...")
    
    url = (
        "https://archive-api.open-meteo.com/v1/era5"
        f"?latitude={lat}&longitude={lon}"
        "&start_date=2015-01-01&end_date=2026-05-06"
        "&daily=temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant"
        "&timezone=Europe/Madrid"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if "daily" in data and "time" in data["daily"]:
            df = pd.DataFrame(data["daily"])
            
            df.rename(columns={
                "temperature_2m_mean": "Temperature",
                "relative_humidity_2m_mean": "Humidity",
                "precipitation_sum": "Precipitation",
                "wind_speed_10m_max": "WindSpeed",
                "wind_direction_10m_dominant": "WindDirection"
            }, inplace=True)
            
            df["Date"] = pd.to_datetime(df["time"])
            df.drop(columns=["time"], inplace=True)
            
            df.to_parquet(f"{name}_weather.parquet", index=False)
            print(f"✅ Success: {name} saved.")
        else:
            print(f"❌ Error: Could not extract data for {name}.")
            
    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
    
    time.sleep(1.5) 

print("\nFinished!")

Processing: ALGORTA_BBIZI2...
✅ Success: ALGORTA_BBIZI2 saved.
Processing: BARAKALDO...
✅ Success: BARAKALDO saved.
Processing: BASAURI...
✅ Success: BASAURI saved.
Processing: ERANDIO...
✅ Success: ERANDIO saved.
Processing: MAZARREDO...
✅ Success: MAZARREDO saved.
Processing: MUSKIZ...
❌ Error: Could not extract data for MUSKIZ.
Processing: SANTURTZI...
❌ Error: Could not extract data for SANTURTZI.

Finished!


In [7]:
import pandas as pd
import requests
import time

stations = {
    "MUSKIZ": (43.323, -3.113),
    "SANTURTZI": (43.323, -3.032)
}

for name, (lat, lon) in stations.items():
    print(f"Processing: {name}...")
    
    url = (
        "https://archive-api.open-meteo.com/v1/era5"
        f"?latitude={lat}&longitude={lon}"
        "&start_date=2015-01-01&end_date=2026-05-06"
        "&daily=temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant"
        "&timezone=Europe/Madrid"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if "daily" in data and "time" in data["daily"]:
            df = pd.DataFrame(data["daily"])
            
            df.rename(columns={
                "temperature_2m_mean": "Temperature",
                "relative_humidity_2m_mean": "Humidity",
                "precipitation_sum": "Precipitation",
                "wind_speed_10m_max": "WindSpeed",
                "wind_direction_10m_dominant": "WindDirection"
            }, inplace=True)
            
            df["Date"] = pd.to_datetime(df["time"])
            df.drop(columns=["time"], inplace=True)
            
            df.to_parquet(f"{name}_weather.parquet", index=False)
            print(f"✅ Success: {name} saved.")
        else:
            print(f"❌ Error: Could not extract data for {name}.")
            
    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
    
    time.sleep(1.5) 

print("\nFinished!")

Processing: MUSKIZ...
✅ Success: MUSKIZ saved.
Processing: SANTURTZI...
✅ Success: SANTURTZI saved.

Finished!


In [8]:
import pandas as pd
from pathlib import Path

folder = Path(".")
all_files = folder.glob("*_weather.parquet")

combined_df = pd.DataFrame()

for file_path in all_files:
    station_name = file_path.stem.replace("_weather", "")
    
    df = pd.read_parquet(file_path)
    
    df["Station"] = station_name
    
    combined_df = pd.concat([combined_df, df], ignore_index=True)

combined_df = combined_df.sort_values(by=["Date", "Station"])

print(combined_df.head())

       Temperature  Humidity  Precipitation  WindSpeed  WindDirection  \
0              6.4        85            0.0        8.0            176   
4144           6.7        85            0.0        8.0            176   
8288           4.6        82            0.0        8.0            176   
12432          6.8        85            0.0        8.0            176   
16576          6.6        83            0.0        8.0            176   

            Date         Station  
0     2015-01-01  ALGORTA_BBIZI2  
4144  2015-01-01       BARAKALDO  
8288  2015-01-01         BASAURI  
12432 2015-01-01         ERANDIO  
16576 2015-01-01       MAZARREDO  


In [11]:
combined_df.info()

<class 'pandas.DataFrame'>
Index: 29008 entries, 0 to 29007
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Temperature    29008 non-null  float64       
 1   Humidity       29008 non-null  int64         
 2   Precipitation  29008 non-null  float64       
 3   WindSpeed      29008 non-null  float64       
 4   WindDirection  29008 non-null  int64         
 5   Date           29008 non-null  datetime64[us]
 6   Station        29008 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(2), str(1)
memory usage: 2.0 MB


In [12]:
combined_df.isnull().sum()

Temperature      0
Humidity         0
Precipitation    0
WindSpeed        0
WindDirection    0
Date             0
Station          0
dtype: int64

In [13]:
combined_df

,Temperature,Humidity,Precipitation,WindSpeed,WindDirection,Date,Station
0,6.4,85,0.0,8.0,176,2015-01-01,ALGORTA_BBIZI2
4144,6.7,85,0.0,8.0,176,2015-01-01,BARAKALDO
8288,4.6,82,0.0,8.0,176,2015-01-01,BASAURI
12432,6.8,85,0.0,8.0,176,2015-01-01,ERANDIO
16576,6.6,83,0.0,8.0,176,2015-01-01,MAZARREDO
...,...,...,...,...,...,...,...
12431,11.4,89,2.6,14.1,288,2026-05-06,BASAURI
16575,12.5,88,6.5,17.7,280,2026-05-06,ERANDIO
20719,11.9,88,2.3,13.3,289,2026-05-06,MAZARREDO
24863,12.7,87,2.9,21.2,281,2026-05-06,MUSKIZ


In [14]:
combined_df.to_parquet("all_stations_combined.parquet", index=False)
print(f"\nSuccessfully combined all {len(combined_df)} records into 'all_stations_combined.parquet'")


Successfully combined all 29008 records into 'all_stations_combined.parquet'
